# Gold Forecast — Notebook de Ciência de Dados
**Objetivo:** prever o preço do ouro com **Random Forest**, prevendo a **variação (retorno) de 1 dia** e reconstruindo o preço.

> **Por que prever a variação, e não o nível do preço?** Modelos de árvore (como o Random Forest) não extrapolam: só devolvem valores dentro da faixa vista no treino. Em uma série com tendência de alta, o período de teste fica acima dessa faixa e o modelo "trava" num teto, gerando R² negativo. Prevendo a **variação** (grandeza estável/estacionária) e reconstruindo o preço com `preço_amanhã = preço_hoje × (1 + variação)`, o problema desaparece.

> **Antes de executar:** baixe o CSV **real** do ouro no [Kaggle](https://www.kaggle.com/datasets/lbronchal/gold-and-silver-prices-dataset) e faça upload no Colab. O notebook inspeciona as colunas para que a coluna de preço correta seja escolhida explicitamente. Este notebook usa **somente o Random Forest** e seleciona o nível de regularização por **validação cruzada temporal** (igual ao `train_model.py`), evitando overfitting.

## 1. Instalação e importação

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from google.colab import files
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path

RANDOM_STATE = 42
N_ARVORES = 300        # árvores do modelo final
N_ARVORES_BUSCA = 150  # árvores durante a busca por CV (mais rápido, mesma escolha)

# Features estacionarias (retornos / volatilidade / posicao relativa a media).
# Nenhuma feature e o "nivel" do preco -> evita o problema de extrapolacao.
FEATURES = [
    "ret_1", "ret_2", "ret_3", "ret_5", "ret_7", "ret_14", "ret_30",
    "ret_media_7", "ret_media_30",
    "volatilidade_7", "volatilidade_30",
    "dist_media_30",
]

# Só Random Forest. A CV escolhe o nível de REGULARIZAÇÃO (anti-overfitting).
GRADE_HIPERPARAMETROS = [
    {"max_depth": d, "min_samples_leaf": l}
    for d in [3, 4, 6, 10]
    for l in [20, 50, 100]
]

## 2. Upload e inspeção dos dados

In [ ]:
uploaded = files.upload()
arquivo = next(iter(uploaded))
df_original = pd.read_csv(arquivo, sep=None, engine="python")  # detecta ',' ou ';'

print("Dimensões:", df_original.shape)
print("\nColunas:", df_original.columns.tolist())
display(df_original.head())
df_original.info()

### Escolha das colunas
Veja a saída acima. Ajuste `COLUNA_DATA` e `COLUNA_PRECO` para os nomes exatos do arquivo. Em bases de ouro pode haver mais de uma cotação ou moeda (ex.: `USD (AM)`, `USD (PM)`): escolha **uma** série única e documente a unidade no relatório.

In [ ]:
COLUNA_DATA = "Date"       # ALTERE se o CSV usar outro nome
COLUNA_PRECO = "Price"     # ALTERE para a coluna de preço correta (ex.: "USD (PM)")

assert COLUNA_DATA in df_original.columns, f"Coluna de data não encontrada: {COLUNA_DATA}"
assert COLUNA_PRECO in df_original.columns, f"Coluna de preço não encontrada: {COLUNA_PRECO}" 

## 3. Limpeza e análise exploratória

In [ ]:
def parsear_datas(serie):
    """Robusto aos dois formatos comuns:
    - ISO 'YYYY-MM-DD' (dataset do Kaggle) -> parse direto, SEM dayfirst
      (usar dayfirst aqui corromperia, trocando mês por dia).
    - 'DD/MM/YYYY' (formato BR/EU) -> dayfirst=True.
    """
    s = serie.astype(str).str.strip()
    padrao_iso = s.str.match(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}")
    if padrao_iso.mean() > 0.8:
        return pd.to_datetime(s, errors="coerce")
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

df = df_original[[COLUNA_DATA, COLUNA_PRECO]].copy()
df.columns = ["Date", "Price"]
df["Date"] = parsear_datas(df["Date"])
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.dropna(subset=["Date", "Price"])
df = df.drop_duplicates(subset=["Date"], keep="last")
df = df.sort_values("Date").reset_index(drop=True)
df = df[df["Price"] > 0].copy()

print("Dimensões após limpeza:", df.shape)
print("Período:", df["Date"].min(), "até", df["Date"].max())
print("Ausentes:\n", df.isna().sum())
display(df.describe())
display(df.head())

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df["Date"], df["Price"])
plt.title("Histórico do preço do ouro")
plt.xlabel("Data")
plt.ylabel("Preço (unidade conforme dataset)")
plt.grid(alpha=.3)
plt.show()

**Interpretação:** observe tendência, períodos de alta/queda e volatilidade. Não confunda ausência de cotação em fins de semana/feriados com dado faltante: são dias sem observação de mercado. Não inventamos preços para esses dias.

## 4. Engenharia de atributos e variável alvo (previsão de variação)

O **alvo** é o retorno do próximo dia: `target_ret = preço_amanhã / preço_hoje − 1`. As **features** são todas derivadas do retorno (defasagens, médias, volatilidade) ou da posição relativa à média — grandezas que não saem da faixa vista no treino. Guardamos `preco_atual` e `target_preco` apenas para **reconstruir** o preço e medir o erro na escala de preço.

In [ ]:
def criar_features(dados):
    d = dados.copy()
    p = d["Price"]
    r = p.pct_change()

    # retornos defasados
    d["ret_1"] = r
    d["ret_2"] = r.shift(1)
    d["ret_3"] = r.shift(2)
    d["ret_5"] = r.shift(4)
    d["ret_7"] = r.shift(6)
    d["ret_14"] = r.shift(13)
    d["ret_30"] = r.shift(29)

    # estatísticas de retorno
    d["ret_media_7"] = r.rolling(7).mean()
    d["ret_media_30"] = r.rolling(30).mean()
    d["volatilidade_7"] = r.rolling(7).std()
    d["volatilidade_30"] = r.rolling(30).std()

    # posição relativa à média de 30 (estacionária)
    d["dist_media_30"] = p / p.rolling(30).mean() - 1

    # alvo = retorno do próximo dia; guardamos preço e data-alvo para reconstruir
    d["preco_atual"] = p
    d["target_ret"] = p.shift(-1) / p - 1
    d["target_preco"] = p.shift(-1)
    d["Target_Date"] = d["Date"].shift(-1)

    return d.dropna().reset_index(drop=True)

dados = criar_features(df)
display(dados[["Date"] + FEATURES + ["target_ret", "target_preco"]].head())
print("Linhas após criação das features:", len(dados))

## 5. Separação temporal: treino e teste

In [ ]:
corte = int(len(dados) * 0.80)
treino = dados.iloc[:corte].copy()
teste = dados.iloc[corte:].copy()

X_treino, y_treino = treino[FEATURES], treino["target_ret"]
X_teste, y_teste = teste[FEATURES], teste["target_ret"]

print("Treino:", treino["Date"].min(), "até", treino["Date"].max(), X_treino.shape)
print("Teste:", teste["Date"].min(), "até", teste["Date"].max(), X_teste.shape)

Não embaralhamos a série. O teste contém as observações mais recentes, simulando o uso do passado para prever o futuro. A avaliação é de um passo à frente: em cada data do teste, os preços anteriores já observados podem ser usados para calcular as features.

## 6. Seleção de hiperparâmetros por validação temporal (somente Random Forest)

Usamos **somente o Random Forest**. O que a validação cruzada escolhe é o **nível de regularização** (profundidade da árvore e tamanho mínimo da folha). Árvores muito profundas decoram ruído dos retornos (*overfitting*); árvores rasas generalizam melhor. A `TimeSeriesSplit` é feita **apenas dentro do treino** e o erro é medido na **escala de preço** (reconstruindo o preço a partir do retorno previsto), para ficar comparável com a avaliação de teste. **Não usamos o teste para escolher a configuração.**

In [ ]:
def _novo_rf(n_estimators=N_ARVORES, **kw):
    return RandomForestRegressor(
        n_estimators=n_estimators, random_state=RANDOM_STATE, n_jobs=-1, **kw
    )

def mae_cv_preco(treino, kw, n_splits):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    X = treino[FEATURES]
    y_ret = treino["target_ret"].to_numpy()
    preco_atual = treino["preco_atual"].to_numpy()
    preco_real = treino["target_preco"].to_numpy()
    maes = []
    for a, b in tscv.split(X):
        m = _novo_rf(n_estimators=N_ARVORES_BUSCA, **kw).fit(X.iloc[a], y_ret[a])
        pp = preco_atual[b] * (1 + m.predict(X.iloc[b]))
        maes.append(mean_absolute_error(preco_real[b], pp))
    return float(np.mean(maes)), float(np.std(maes))

n_splits = 5 if len(treino) >= 500 else 3
linhas = []
for kw in GRADE_HIPERPARAMETROS:
    mae, desvio = mae_cv_preco(treino, kw, n_splits)
    linhas.append({
        "Modelo": f"Random Forest (depth={kw['max_depth']}, leaf={kw['min_samples_leaf']})",
        "max_depth": kw["max_depth"], "min_samples_leaf": kw["min_samples_leaf"],
        "MAE_CV": mae, "Desvio_CV": desvio
    })

cv_df = pd.DataFrame(linhas).sort_values("MAE_CV").reset_index(drop=True)
melhor_kw = {"max_depth": int(cv_df.iloc[0]["max_depth"]),
             "min_samples_leaf": int(cv_df.iloc[0]["min_samples_leaf"])}
print("Config escolhida pela CV:", melhor_kw)
display(cv_df)

modelo = _novo_rf(**melhor_kw)

## 7. Avaliação final no teste e baseline

In [ ]:
def metricas_regressao(y_real, y_previsto):
    return {
        "MAE": mean_absolute_error(y_real, y_previsto),
        "MSE": mean_squared_error(y_real, y_previsto),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_previsto)),
        "R2": r2_score(y_real, y_previsto)
    }

modelo.fit(X_treino, y_treino)

# Previsão no teste: retorno -> reconstrução do preço
ret_pred = modelo.predict(X_teste)
preco_atual_te = teste["preco_atual"].to_numpy()
preco_real_te = teste["target_preco"].to_numpy()
preco_pred_te = preco_atual_te * (1 + ret_pred)

metricas_modelo = metricas_regressao(preco_real_te, preco_pred_te)
# Baseline ingênuo: amanhã = último preço conhecido (retorno zero)
metricas_baseline = metricas_regressao(preco_real_te, preco_atual_te)

comparacao = pd.DataFrame([
    {"Modelo": "Random Forest", **metricas_modelo},
    {"Modelo": "Baseline (último preço)", **metricas_baseline}
])
display(comparacao)

## 8. Gráfico: real x previsto

In [ ]:
resultado_teste = pd.DataFrame({
    "Date": teste["Target_Date"].to_numpy(),
    "Real": preco_real_te,
    "Previsto": preco_pred_te
})

plt.figure(figsize=(15,5))
plt.plot(resultado_teste["Date"], resultado_teste["Real"], label="Real")
plt.plot(resultado_teste["Date"], resultado_teste["Previsto"], label="Previsto")
plt.title("Preço real x previsto — conjunto de teste")
plt.xlabel("Data")
plt.ylabel("Preço")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**Como interpretar:** MAE e RMSE estão na mesma unidade do preço; quanto menores, melhor. MSE penaliza mais os erros grandes. R² próximo de 1 indica que o modelo explica boa parte da variação; pode ser negativo. Compare o modelo com o baseline: se não superar a referência simples, não devemos afirmar que ele agrega poder preditivo.

## 9. Persistência do modelo e artefatos

In [ ]:
Path("models").mkdir(exist_ok=True)

nome_completo = f"Random Forest (depth={melhor_kw['max_depth']}, leaf={melhor_kw['min_samples_leaf']})"

bundle = {
    "model": modelo,
    "model_name": nome_completo,
    "best_params": melhor_kw,
    "features": FEATURES,
    "history": df,
    "price_column_original": COLUNA_PRECO,
    "date_column_original": COLUNA_DATA,
    "metrics": {k: float(v) for k, v in metricas_modelo.items()},
    "baseline_metrics": {k: float(v) for k, v in metricas_baseline.items()},
    "cv_results": cv_df,
    "test_predictions": resultado_teste.rename(columns={"Previsto": "Predicted"}),
    "training_cutoff": str(treino["Date"].max().date()),
    "test_start": str(teste["Date"].min().date()),
    "test_end": str(teste["Date"].max().date())
}

joblib.dump(bundle, "models/gold_forecast_bundle.joblib")
cv_df.to_csv("models/comparacao_validacao_temporal.csv", index=False)
comparacao.to_csv("models/metricas_teste.csv", index=False)
bundle["test_predictions"].to_csv("models/previsoes_teste.csv", index=False)
df.to_csv("models/historico_limpo.csv", index=False)

print("Modelo selecionado:", nome_completo)
print("Artefatos salvos.")

> **Compatibilidade com o Streamlit:** o `test_predictions` é salvo com as colunas `Date`, `Real`, `Predicted` (o `app.py` espera `Predicted`). As chaves do `bundle` são as mesmas que o `train_model.py` gera, então o dashboard carrega o modelo sem alterações.

## 10. Download do modelo para usar no Streamlit

In [ ]:
!zip -r gold_forecast_modelos.zip models
files.download("gold_forecast_modelos.zip")

## Conclusão do notebook

Registre no relatório os valores **efetivamente calculados** de MAE, MSE, RMSE e R², as datas do treino/teste e se o Random Forest superou o baseline. Não invente resultados: eles dependem do CSV real e da coluna de preço selecionada. Este notebook usa apenas o histórico do preço — não considera câmbio, juros, inflação ou eventos externos.